# Perbandingan Metode Optimasi Hiperparameter dan Interpretasi SHAP pada LightGBM untuk Prediksi Kepatuhan Pengobatan Pasien Penyakit Tidak Menular

**Ihya' Nashirudin Abrar** — Magister Informatika, Universitas Ahmad Dahlan (2026)

Notebook ini memuat **seluruh eksperimen tesis dari awal sampai akhir** dalam satu alur yang dapat dijalankan ulang:

| Bagian | Isi | Rujukan di tesis |
|---|---|---|
| 1 | Setup, pustaka, dan *random seed* | Subbab 3.2.1 |
| 2 | Pemuatan data dan eksplorasi | Subbab 3.2.2, 4.1 |
| 3 | Pra-pemrosesan (deduplikasi, *encoding*, *feature engineering*, *stratified split*) | Subbab 3.3.2, 4.1 |
| 4 | Studi pendahuluan: lima algoritma *boosting* (justifikasi pemilihan LightGBM) | Lampiran 2 |
| 5 | **Tahap 1** — Optimasi hiperparameter (6 metode) + uji Friedman dan Nemenyi | Subbab 3.3.3, 4.2 |
| 6 | Pelatihan model final | Subbab 4.2 |
| 7 | **Tahap 2** — Interpretasi SHAP empat level | Subbab 3.3.4, 4.3 |
| 8 | Evaluasi kinerja model final (per kelas, IK 95% *bootstrap*, Youden eksploratif) | Subbab 3.3.5, 4.4 |
| 9 | Analisis sensitivitas (pembobotan kelas, ablasi fitur unit klaim, *baseline* vs final) | Subbab 4.5 |
| 10 | Ringkasan | Bab 5 |

Seluruh komponen stokastik memakai `random_state = 42`. Waktu eksekusi penuh sekitar 35–45 menit (CPU), terutama pada Tahap 1.

## 1. Setup dan Pustaka

In [1]:
import warnings, time, json, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, GridSearchCV,
                                      RandomizedSearchCV)
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, roc_curve, auc,
                             classification_report)
from scipy import stats
from scipy.stats import randint, uniform, loguniform

from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import shap

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import BayesSearchCV
from skopt.space import Integer, Real
import scikit_posthocs as sp

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'Final Prepared Dataset - Diabetes and Hypertension Data.xlsx'
FIG = str(ROOT / 'figures')
OUT = str(ROOT / 'outputs')
os.makedirs(FIG, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# konsistensi tema visual mengikuti gaya paper
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titleweight': 'bold', 'axes.titlesize': 13,
    'figure.facecolor': 'white', 'savefig.bbox': 'tight',
})
PALETTE = ['#2E5A87', '#C0504D', '#4F8A5B', '#E39A2C', '#7B5EA7']
import sys, sklearn, lightgbm, xgboost, catboost, scipy
print('Pustaka siap. Versi kunci:')
print(f'  python={sys.version.split()[0]} sklearn={sklearn.__version__} lightgbm={lightgbm.__version__} '
      f'optuna={optuna.__version__} shap={shap.__version__} scipy={scipy.__version__}')
print(f'  xgboost={xgboost.__version__} catboost={catboost.__version__}')

Pustaka siap. Versi kunci:
  python=3.11.1 sklearn=1.8.0 lightgbm=4.6.0 optuna=4.8.0 shap=0.49.1 scipy=1.15.3
  xgboost=1.6.2 catboost=1.2.10


## 2. Pemuatan Data dan Eksplorasi

Dataset publik *Final Prepared Dataset — Diabetes and Hypertension Data* (Kanyongo, 2024; Kanyongo et al., 2025b), lisensi CC0 1.0, DOI [10.17632/zkp7sbbx64.2](https://doi.org/10.17632/zkp7sbbx64.2). Target **ADHERENCE** ditentukan dari frekuensi pengambilan ulang (*refill*) obat bulanan dalam 12 bulan: **ADHERENT** = 9–12 kali (≥ 75%), **NON-ADHERENT** = 1–8 kali.

In [2]:
df = pd.read_excel(DATA_PATH)
print('Ukuran data mentah :', df.shape)
print('Kolom              :', list(df.columns))
print('\nDistribusi target ADHERENCE:')
print(df['ADHERENCE'].value_counts())
print('\nProporsi (%):')
print((df['ADHERENCE'].value_counts(normalize=True) * 100).round(2))
df.head()

Ukuran data mentah : (24084, 12)
Kolom              : ['ADHERENCE', 'AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION']

Distribusi target ADHERENCE:
ADHERENCE
NON-ADHERENT    14415
ADHERENT         9669
Name: count, dtype: int64

Proporsi (%):
ADHERENCE
NON-ADHERENT    59.85
ADHERENT        40.15
Name: proportion, dtype: float64


,ADHERENCE,AGE,ANNUALCONTRIBUTION,ANNUALCLAIMAMOUNT,UNITSTOTAL,GENDER_M,SCHEMETYPE_MEDIUM,SCHEMETYPE_PREMIUM,DIAGNOSIS_HYPERTENSION,COVERTYPE_STANDARD,COMORBIDITY_NO_COMORBIDITY,COMPLICATIONDEVELOPMENT_NO_COMPLICATION
0,NON-ADHERENT,24,690384.0,10400.00,60.0,False,False,False,False,True,True,True
1,NON-ADHERENT,53,2718696.0,306366.53,1380.0,True,True,False,False,True,True,True
2,NON-ADHERENT,46,808284.0,77480.00,600.0,True,False,False,False,True,False,True
3,NON-ADHERENT,76,808284.0,198800.58,840.0,True,False,False,False,True,False,True
4,ADHERENT,69,808284.0,116872.04,720.0,False,False,False,False,True,False,True


In [3]:
# --- Gambar: distribusi target ---
counts = df['ADHERENCE'].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
bars = ax[0].bar(counts.index, counts.values, color=[PALETTE[1], PALETTE[0]], edgecolor='black', width=0.6)
ax[0].set_title('Distribusi Kelas Target')
ax[0].set_ylabel('Jumlah Pasien')
for b, v in zip(bars, counts.values):
    ax[0].text(b.get_x()+b.get_width()/2, v+150, f'{v:,}\n({v/counts.sum()*100:.1f}%)',
               ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].set_ylim(0, counts.max()*1.18)
ax[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90,
          colors=[PALETTE[1], PALETTE[0]], wedgeprops=dict(edgecolor='white', linewidth=2),
          textprops=dict(fontsize=11))
ax[1].set_title('Proporsi Kelas Target')
plt.tight_layout(); plt.savefig(f'{FIG}/fig01_distribusi_target.png'); plt.show()
imbalance = counts.max()/counts.min()
print(f'Rasio ketidakseimbangan = {imbalance:.2f} : 1')

Rasio ketidakseimbangan = 1.49 : 1


## 3. Pra-Pemrosesan Data

1. Deduplikasi baris eksak.
2. *Encoding* target `ADHERENT = 1`, `NON-ADHERENT = 0`; fitur boolean dikonversi ke integer 1/0.
3. *Feature engineering* per baris (tanpa statistik agregat, sehingga aman dilakukan sebelum *split*): `LOG_CLAIM`, `LOG_UNITS`, `LOG_CONTRIB` (log1p), `CLAIM_PER_UNIT`, `CLAIM_RATIO` (rasio + log1p), dan `UNITS_PER_AGE` → **17 fitur**.
4. *Stratified split* 80:20.

Tidak dilakukan normalisasi (LightGBM berbasis pohon), tidak ada pembobotan kelas, dan tidak ada SMOTE pada model utama. Pengaruh pembobotan kelas diuji terpisah pada Bagian 9.

In [4]:
n_dup = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f'Baris duplikat dihapus : {n_dup}  ->  sisa {df.shape[0]} rekaman unik')

df_p = df.copy()
df_p['ADHERENCE'] = (df_p['ADHERENCE'] == 'ADHERENT').astype(int)
for c in df_p.columns:
    if df_p[c].dtype == 'bool':
        df_p[c] = df_p[c].astype(int)

y = df_p['ADHERENCE']
X = df_p.drop(columns=['ADHERENCE']).copy()

# --- feature engineering (Paper 2) ---
X['LOG_CLAIM']      = np.log1p(X['ANNUALCLAIMAMOUNT'])
X['LOG_UNITS']      = np.log1p(X['UNITSTOTAL'])
X['LOG_CONTRIB']    = np.log1p(X['ANNUALCONTRIBUTION'])
X['CLAIM_PER_UNIT'] = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['UNITSTOTAL'] + 1))
X['CLAIM_RATIO']    = np.log1p(X['ANNUALCLAIMAMOUNT'] / (X['ANNUALCONTRIBUTION'] + 1))
X['UNITS_PER_AGE']  = X['UNITSTOTAL'] / (X['AGE'] + 1)

print(f'Jumlah fitur setelah engineering : {X.shape[1]}')
print('Fitur:', list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f'\nTrain: {X_train.shape[0]} sampel | Test: {X_test.shape[0]} sampel')
print('Proporsi kelas train:', dict(y_train.value_counts(normalize=True).round(3)))

Baris duplikat dihapus : 13  ->  sisa 24071 rekaman unik
Jumlah fitur setelah engineering : 17
Fitur: ['AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL', 'GENDER_M', 'SCHEMETYPE_MEDIUM', 'SCHEMETYPE_PREMIUM', 'DIAGNOSIS_HYPERTENSION', 'COVERTYPE_STANDARD', 'COMORBIDITY_NO_COMORBIDITY', 'COMPLICATIONDEVELOPMENT_NO_COMPLICATION', 'LOG_CLAIM', 'LOG_UNITS', 'LOG_CONTRIB', 'CLAIM_PER_UNIT', 'CLAIM_RATIO', 'UNITS_PER_AGE']

Train: 19256 sampel | Test: 4815 sampel
Proporsi kelas train: {0: 0.598, 1: 0.402}


In [5]:
# --- Gambar: korelasi antar fitur ---
plt.figure(figsize=(11, 9))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, annot=False,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.7, 'label': 'Koefisien Korelasi'})
plt.title('Matriks Korelasi Antar Fitur (17 Fitur)')
plt.tight_layout(); plt.savefig(f'{FIG}/fig02_korelasi_fitur.png'); plt.show()